# General Creators: YouTube API Data Collection

Collect seed channels, video IDs, and video metadata for the general creator sample.


##0. Create topic and define producer

In [14]:
# SKIP THIS
from kafka import KafkaProducer

topic = "youtube_video_raw"
bootstrap_server = "localhost:9092"

producer = KafkaProducer(
    bootstrap_servers=[bootstrap_server]
)
print("connected")

connected


##1. Youtube API
**1.1 Fetch seed channels**

In [11]:
API_KEY = "YOUR_YOUTUBE_API_KEY"
youtube = build("youtube", "v3", developerKey=API_KEY)

OUTPUT_DIR = "/content/drive/MyDrive/Classes/MIS 584/Project Colab Output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

START_DATE = "2010-01-01T00:00:00Z"
END_DATE   = "2011-12-31T23:59:59Z"

TARGET_CHANNELS = 500

SEARCH_QUERIES = [
    "music", "news", "tutorial", "vlog", "game",
    "review", "interview", "comedy", "sports", "education"
]

### Checkpoint & Retry Helpers

These utilities allow long-running API collection jobs to resume from where they left off after a Colab disconnect or quota error, and add exponential-backoff retry logic to every API call.

In [12]:
# ─────────────────────────────────────────────
# Checkpoint helpers
# ─────────────────────────────────────────────
CHECKPOINT_DIR = "/content/drive/MyDrive/Classes/MIS 584/Project Colab Output/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

def save_checkpoint(name: str, data) -> None:
    """Persist a JSON-serialisable object to disk."""
    path = os.path.join(CHECKPOINT_DIR, f"{name}.json")
    tmp  = path + ".tmp"
    with open(tmp, "w") as f:
        json.dump(data, f)
    os.replace(tmp, path)          # atomic write — avoids corrupt files on crash
    print(f"  ✓ checkpoint saved → {path}")

def load_checkpoint(name: str):
    """Return the saved object, or None if no checkpoint exists yet."""
    path = os.path.join(CHECKPOINT_DIR, f"{name}.json")
    if os.path.exists(path):
        with open(path) as f:
            data = json.load(f)
        print(f"  ↩  resuming from checkpoint → {path}")
        return data
    return None

def clear_checkpoint(name: str) -> None:
    """Delete a checkpoint once a stage has finished successfully."""
    path = os.path.join(CHECKPOINT_DIR, f"{name}.json")
    if os.path.exists(path):
        os.remove(path)
        print(f"  🗑  checkpoint cleared → {path}")


# ─────────────────────────────────────────────
# API call optimisation: exponential backoff
# ─────────────────────────────────────────────
def api_call_with_retry(request_fn, max_retries: int = 6):
    """
    Execute a YouTube API request with exponential back-off.

    Parameters
    ----------
    request_fn : callable
        A zero-argument lambda that returns a googleapiclient Request object,
        e.g.  lambda: youtube.videos().list(...)
        Passing a factory (not the executed request) lets us rebuild the
        request on each retry so the HTTP connection is fresh.
    max_retries : int
        Maximum number of attempts before re-raising the last exception.

    Retry behaviour
    ---------------
    - 403 / 429  quota / rate-limit → exponential back-off (2^n + jitter seconds)
    - 5xx        server error       → shorter back-off
    - 4xx other  bad request        → raise immediately (retrying won't help)
    """
    import random
    last_exc = None
    for attempt in range(max_retries):
        try:
            return request_fn().execute()
        except HttpError as e:
            status = e.resp.status
            last_exc = e
            if status in (403, 429):
                wait = (2 ** attempt) + random.uniform(0, 1)
                print(f"  ⏳ quota/rate-limit (HTTP {status}), attempt {attempt+1}/{max_retries}. "                      f"Waiting {wait:.1f}s …")
                time.sleep(wait)
            elif status >= 500:
                wait = 2 ** attempt
                print(f"  ⚠  server error (HTTP {status}), attempt {attempt+1}/{max_retries}. "                      f"Waiting {wait:.0f}s …")
                time.sleep(wait)
            else:
                raise   # 400, 404, etc. — retrying won't fix these
    raise last_exc

**1.1 fetch_seed_channels — with checkpointing & retry**

In [13]:
def fetch_seed_channels(target_channels: int = 300) -> pd.DataFrame:
    """
    Collect unique seed channels by searching for videos in the study window.
    Checkpoints after every API page so the job can resume after interruption.
    """
    # ── Resume from checkpoint if one exists ──
    existing = load_checkpoint("seed_channels")
    rows          = existing if existing else []
    collected_ids = {r["channel_id"] for r in rows}

    per_query = math.ceil(target_channels / len(SEARCH_QUERIES) * 2)

    for q in tqdm(SEARCH_QUERIES, desc="search channels"):
        next_page_token = None
        count = 0

        while count < per_query:
            try:
                response = api_call_with_retry(
                    lambda q=q, pt=next_page_token: youtube.search().list(
                        part="snippet",
                        q=q,
                        type="video",
                        maxResults=50,
                        pageToken=pt,
                        publishedAfter=START_DATE,
                        publishedBefore=END_DATE
                    )
                )
            except Exception as e:
                print(f"  ✗ Fatal error on query '{q}': {e}. Progress saved.")
                save_checkpoint("seed_channels", rows)
                raise

            for item in response.get("items", []):
                cid = item["snippet"].get("channelId")
                if not cid or cid in collected_ids:
                    continue
                rows.append({
                    "channel_id":    cid,
                    "channel_title": item["snippet"].get("channelTitle", "")
                })
                collected_ids.add(cid)
                count += 1
                if count >= per_query:
                    break

            # ── Save after every page ──
            save_checkpoint("seed_channels", rows)

            next_page_token = response.get("nextPageToken")
            if not next_page_token:
                break

            time.sleep(0.1)   # polite delay between pages

    channel_df = (
        pd.DataFrame(rows)
          .drop_duplicates(subset=["channel_id"])
          .head(target_channels)
    )
    channel_df.to_csv(f"{OUTPUT_DIR}/seed_channels.csv", index=False)
    print(f"Seed channels collected: {len(channel_df)}")
    return channel_df

In [ ]:
channel_df = fetch_seed_channels(TARGET_CHANNELS)
channel_df.head()

search channels:   0%|          | 0/10 [00:00<?, ?it/s]

  ✓ checkpoint saved → /content/drive/MyDrive/Classes/MIS 584/Project Colab Output/checkpoints/seed_channels.json
  ✓ checkpoint saved → /content/drive/MyDrive/Classes/MIS 584/Project Colab Output/checkpoints/seed_channels.json
  ✓ checkpoint saved → /content/drive/MyDrive/Classes/MIS 584/Project Colab Output/checkpoints/seed_channels.json


search channels:  10%|█         | 1/10 [00:02<00:26,  2.93s/it]

  ✓ checkpoint saved → /content/drive/MyDrive/Classes/MIS 584/Project Colab Output/checkpoints/seed_channels.json
  ✓ checkpoint saved → /content/drive/MyDrive/Classes/MIS 584/Project Colab Output/checkpoints/seed_channels.json
  ✓ checkpoint saved → /content/drive/MyDrive/Classes/MIS 584/Project Colab Output/checkpoints/seed_channels.json
  ✓ checkpoint saved → /content/drive/MyDrive/Classes/MIS 584/Project Colab Output/checkpoints/seed_channels.json


search channels:  20%|██        | 2/10 [00:05<00:21,  2.71s/it]

...

  ✓ checkpoint saved → /content/drive/MyDrive/Classes/MIS 584/Project Colab Output/checkpoints/seed_channels.json
  ✓ checkpoint saved → /content/drive/MyDrive/Classes/MIS 584/Project Colab Output/checkpoints/seed_channels.json
  ✓ checkpoint saved → /content/drive/MyDrive/Classes/MIS 584/Project Colab Output/checkpoints/seed_channels.json
  ✓ checkpoint saved → /content/drive/MyDrive/Classes/MIS 584/Project Colab Output/checkpoints/seed_channels.json


search channels: 100%|██████████| 10/10 [00:36<00:00,  3.69s/it]

  ✓ checkpoint saved → /content/drive/MyDrive/Classes/MIS 584/Project Colab Output/checkpoints/seed_channels.json


  🗑  checkpoint cleared → /content/drive/MyDrive/Classes/MIS 584/Project Colab Output/checkpoints/seed_channels.json
Seed channels collected: 500


,channel_id,channel_title
0,UCGnjeahCJW1AF34HBmQTJ-Q,shakiraVEVO
1,UCoUM-UJ7rirJYP8CQ0EIaHA,Bruno Mars
2,UC2gMECGMn5TVbRN5S5tKb8Q,Christina Perri
3,UCsRM0YB_dabtEPGPTKo-gcw,Adele
4,UC3sj1O535VXDiec8YLn2sDg,TheFugeesVEVO


**1.2 Get video list by channel — with checkpointing & retry**

In [14]:
import os
import json
import pandas as pd

def inspect_pipeline_state(output_dir: str):
    print("=== PIPELINE STATE CHECK ===\n")

    # ---- File paths ----
    final_csv_path   = f"{output_dir}/video_ids_by_channel.csv"
    checkpoint_vids  = f"{output_dir}/checkpoints/video_ids.json"
    checkpoint_done  = f"{output_dir}/checkpoints/done_channels.json"

    # ---- Final CSV ----
    if os.path.exists(final_csv_path):
        df = pd.read_csv(final_csv_path)
        print(f"✔ Final CSV exists: {final_csv_path}")
        print(f"   Rows: {len(df)} | Unique video_ids: {df['video_id'].nunique()}")
    else:
        print(f"✗ Final CSV missing: {final_csv_path}")

    print()

    # ---- video_ids checkpoint ----
    if os.path.exists(checkpoint_vids):
        with open(checkpoint_vids, "r") as f:
            data = json.load(f)
        print(f"✔ video_ids checkpoint exists")
        print(f"   Rows stored: {len(data)}")
    else:
        print("✗ video_ids checkpoint missing")

    print()

    # ---- done_channels checkpoint ----
    if os.path.exists(checkpoint_done):
        with open(checkpoint_done, "r") as f:
            data = json.load(f)
        print(f"✔ done_channels checkpoint exists")
        print(f"   Channels completed: {len(data)}")
    else:
        print("✗ done_channels checkpoint missing")

    print("\n=== INTERPRETATION ===")

    if os.path.exists(final_csv_path):
        print("→ You already have a completed dataset. No API calls needed unless force_refresh=True.")
    elif os.path.exists(checkpoint_vids) or os.path.exists(checkpoint_done):
        print("→ Partial progress detected. Resume from checkpoints.")
    else:
        print("→ No saved progress. Full API run required.")

In [15]:
def get_uploads_playlists(channel_ids: List[str]) -> pd.DataFrame:
    """Batch-fetch uploads-playlist IDs for a list of channels (50 per request)."""
    rows = []
    for i in range(0, len(channel_ids), 50):
        batch = channel_ids[i:i+50]
        response = api_call_with_retry(
            lambda b=batch: youtube.channels().list(
                part="contentDetails,snippet",
                id=",".join(b),
                maxResults=50
            )
        )
        for item in response.get("items", []):
            rows.append({
                "channel_id":           item["id"],
                "channel_title":        item["snippet"]["title"],
                "uploads_playlist_id":  item["contentDetails"]["relatedPlaylists"]["uploads"]
            })
    return pd.DataFrame(rows)

In [16]:
def get_video_ids_from_uploads(
    channel_id: str,
    channel_title: str,
    uploads_playlist_id: str
) -> List[dict]:
    """Page through a channel's uploads playlist and collect all video IDs."""
    rows            = []
    next_page_token = None

    while True:
        response = api_call_with_retry(
            lambda pt=next_page_token: youtube.playlistItems().list(
                part="contentDetails,snippet",
                playlistId=uploads_playlist_id,
                maxResults=50,
                pageToken=pt
            )
        )
        items = response.get("items", [])
        if not items:
            break

        for item in items:
            video_id    = item["contentDetails"].get("videoId")
            published_at = (
                item["contentDetails"].get("videoPublishedAt")
                or item["snippet"].get("publishedAt")
            )
            rows.append({
                "channel_id":           channel_id,
                "channel_title":        channel_title,
                "video_id":             video_id,
                "playlist_published_at": published_at
            })

        next_page_token = response.get("nextPageToken")
        if not next_page_token:
            break

    return rows

In [17]:
def collect_video_ids_by_channel(channel_df: pd.DataFrame) -> pd.DataFrame:
    """
    For each channel, page through its uploads playlist to collect video IDs.
    Checkpoints per-channel so interrupted jobs resume without re-fetching
    already-completed channels.
    """
    uploads_df = get_uploads_playlists(channel_df["channel_id"].tolist())

    # ── Resume from checkpoint ──
    done_channels = set(load_checkpoint("done_channels") or [])
    all_rows      = load_checkpoint("video_ids") or []

    for _, row in tqdm(uploads_df.iterrows(), total=len(uploads_df), desc="playlistItems.list"):
        cid = row["channel_id"]
        if cid in done_channels:
            continue   # already fetched this channel in a previous run

        try:
            new_rows = get_video_ids_from_uploads(
                cid, row["channel_title"], row["uploads_playlist_id"]
            )
            all_rows.extend(new_rows)
            done_channels.add(cid)

            # ── Save after every channel ──
            save_checkpoint("video_ids",     all_rows)
            save_checkpoint("done_channels", list(done_channels))

        except Exception as e:
            # Non-fatal: log and skip the channel, keep going
            print(f"  ✗ Skipping channel {cid} ({row['channel_title']}): {e}")
            save_checkpoint("video_ids",     all_rows)
            save_checkpoint("done_channels", list(done_channels))
            continue

    video_list_df = pd.DataFrame(all_rows).drop_duplicates(subset=["video_id"])
    video_list_df.to_csv(f"{OUTPUT_DIR}/video_ids_by_channel.csv", index=False)

    print(f"Video IDs collected: {len(video_list_df)}")
    return video_list_df

In [ ]:
video_list_df = collect_video_ids_by_channel(channel_df)
video_list_df.head()

  ↩  resuming from checkpoint → /content/drive/MyDrive/Classes/MIS 584/Project Colab Output/checkpoints/done_channels.json
  ↩  resuming from checkpoint → /content/drive/MyDrive/Classes/MIS 584/Project Colab Output/checkpoints/video_ids.json


playlistItems.list:   0%|          | 2/500 [00:00<03:22,  2.47it/s]

  ✓ checkpoint saved → /content/drive/MyDrive/Classes/MIS 584/Project Colab Output/checkpoints/video_ids.json
  ✓ checkpoint saved → /content/drive/MyDrive/Classes/MIS 584/Project Colab Output/checkpoints/done_channels.json
  ✓ checkpoint saved → /content/drive/MyDrive/Classes/MIS 584/Project Colab Output/checkpoints/video_ids.json
  ✓ checkpoint saved → /content/drive/MyDrive/Classes/MIS 584/Project Colab Output/checkpoints/done_channels.json


playlistItems.list:   1%|          | 3/500 [00:01<02:48,  2.94it/s]

  ✓ checkpoint saved → /content/drive/MyDrive/Classes/MIS 584/Project Colab Output/checkpoints/video_ids.json
  ✓ checkpoint saved → /content/drive/MyDrive/Classes/MIS 584/Project Colab Output/checkpoints/done_channels.json


...

  ✓ checkpoint saved → /content/drive/MyDrive/Classes/MIS 584/Project Colab Output/checkpoints/video_ids.json
  ✓ checkpoint saved → /content/drive/MyDrive/Classes/MIS 584/Project Colab Output/checkpoints/done_channels.json


playlistItems.list:   4%|▍         | 22/500 [00:40<19:31,  2.45s/it]

  ✓ checkpoint saved → /content/drive/MyDrive/Classes/MIS 584/Project Colab Output/checkpoints/video_ids.json
  ✓ checkpoint saved → /content/drive/MyDrive/Classes/MIS 584/Project Colab Output/checkpoints/done_channels.json


playlistItems.list:   4%|▍         | 22/500 [01:06<24:01,  3.02s/it]


KeyboardInterrupt: 

In [18]:
import os
import pandas as pd

video_ids_path = f"{OUTPUT_DIR}/video_ids_by_channel.csv"

if not os.path.exists(video_ids_path):
    raise FileNotFoundError(f"Could not find: {video_ids_path}")

video_list_df = pd.read_csv(video_ids_path)

# Clean / validate
if "video_id" not in video_list_df.columns:
    raise ValueError(f"'video_id' column not found. Available columns: {list(video_list_df.columns)}")

video_list_df = (
    video_list_df
    .dropna(subset=["video_id"])
    .drop_duplicates(subset=["video_id"])
    .copy()
)

video_list_df["video_id"] = video_list_df["video_id"].astype(str)

print(f"Loaded video_list_df from: {video_ids_path}")
print(f"Rows: {len(video_list_df):,}")
print(f"Unique video IDs: {video_list_df['video_id'].nunique():,}")

video_list_df.head()

Loaded video_list_df from: /content/drive/MyDrive/Classes/MIS 584/Project Colab Output/video_ids_by_channel.csv
Rows: 1,270,267
Unique video IDs: 1,270,267


,channel_id,channel_title,video_id,playlist_published_at
0,UC-8Q-hLdECwQmaWNwXitYDw,KatyPerryVEVO,AzP-Dlek7ZQ,2026-04-21T16:00:26Z
1,UC-8Q-hLdECwQmaWNwXitYDw,KatyPerryVEVO,2kelRnwRA4w,2025-11-07T00:00:07Z
2,UC-8Q-hLdECwQmaWNwXitYDw,KatyPerryVEVO,wYacBm5H33I,2024-12-20T05:00:44Z
3,UC-8Q-hLdECwQmaWNwXitYDw,KatyPerryVEVO,pD_0rDdY91M,2024-12-20T05:00:42Z
4,UC-8Q-hLdECwQmaWNwXitYDw,KatyPerryVEVO,ToehKIx5e78,2024-12-20T05:00:30Z


**1.3 Get video details — optimised & checkpointed**

In [19]:
def parse_duration_to_seconds(duration_str) -> int | None:
    if pd.isna(duration_str):
        return None
    try:
        return int(isodate.parse_duration(duration_str).total_seconds())
    except Exception:
        return None

In [20]:
def publish_to_kafka(payload: Dict) -> None:
    if producer is not None:
        producer.send(KAFKA_TOPIC, payload)

In [21]:
def get_video_details(video_ids: List[str]) -> pd.DataFrame:
    """
    Fetch snippet + contentDetails + statistics for up to 50 video IDs per
    request (the API maximum).

    Optimisations vs. original
    --------------------------
    1. Dropped `status` from `part=` — saves ~1 quota unit per 50-video batch
       because privacy_status filtering already happens at playlist stage.
    2. Exponential-backoff retry via api_call_with_retry().
    3. Per-batch checkpointing: completed batches are saved so a mid-run
       quota exhaustion doesn't discard hours of work.
    4. 50 ms polite delay between batches to avoid triggering rate limits.
    """
    video_ids = pd.Series(video_ids).dropna().drop_duplicates().tolist()

    # ── Resume: find which batches are already done ──
    saved_rows      = load_checkpoint("video_details_rows") or []
    saved_video_ids = {r["video_id"] for r in saved_rows}
    remaining_ids   = [v for v in video_ids if v not in saved_video_ids]
    print(f"  Total IDs: {len(video_ids):,} | Already fetched: {len(saved_video_ids):,} "          f"| Remaining: {len(remaining_ids):,}")

    rows = saved_rows  # start from whatever we already have

    for i in tqdm(range(0, len(remaining_ids), 50), desc="videos.list"):
        batch = remaining_ids[i:i+50]

        try:
            response = api_call_with_retry(
                lambda b=batch: youtube.videos().list(
                    part="snippet,contentDetails,statistics",  # dropped 'status' — saves quota
                    id=",".join(b),
                    maxResults=50
                )
            )
        except Exception as e:
            print(f"  ✗ Batch {i//50} failed after retries: {e}. Progress saved.")
            save_checkpoint("video_details_rows", rows)
            raise

        for item in response.get("items", []):
            snippet = item.get("snippet", {})
            content = item.get("contentDetails", {})
            stats   = item.get("statistics", {})

            rows.append({
                "video_id":             item.get("id"),
                "actual_channel_id":    snippet.get("channelId"),
                "actual_channel_title": snippet.get("channelTitle"),
                "title":                snippet.get("title"),
                "actual_published_at":  snippet.get("publishedAt"),
                "duration_iso8601":     content.get("duration"),
                "duration_seconds":     parse_duration_to_seconds(content.get("duration")),
                "view_count":           pd.to_numeric(stats.get("viewCount"),   errors="coerce"),
                "like_count":           pd.to_numeric(stats.get("likeCount"),   errors="coerce"),
                "comment_count":        pd.to_numeric(stats.get("commentCount"), errors="coerce"),
                # privacy_status not returned (dropped 'status' part) — will be set to
                # 'public' downstream since only public videos appear in playlist results
                "privacy_status":       "public"
            })

        # ── Save after every batch ──
        save_checkpoint("video_details_rows", rows)
        time.sleep(0.05)  # 50 ms polite delay


    return pd.DataFrame(rows)

In [22]:
from typing import List
def get_video_details(video_ids: List[str]) -> pd.DataFrame:
    """
    Fetch snippet + contentDetails + statistics for up to 50 video IDs per
    request (the API maximum).

    Optimisations vs. original
    --------------------------
    1. Dropped `status` from `part=` — saves ~1 quota unit per 50-video batch
       because privacy_status filtering already happens at playlist stage.
    2. Exponential-backoff retry via api_call_with_retry().
    3. Per-batch checkpointing: completed batches are saved so a mid-run
       quota exhaustion doesn’t discard hours of work.
    4. 50 ms polite delay between batches to avoid triggering rate limits.
    """
    video_ids = pd.Series(video_ids).dropna().drop_duplicates().tolist()

    # ── Resume: find which batches are already done ──
    saved_rows      = load_checkpoint("video_details_rows") or []
    saved_video_ids = {r["video_id"] for r in saved_rows}
    remaining_ids   = [v for v in video_ids if v not in saved_video_ids]
    print(f"  Total IDs: {len(video_ids):,} | Already fetched: {len(saved_video_ids):,} "          f"| Remaining: {len(remaining_ids):,}")

    rows = saved_rows  # start from whatever we already have

    # Helper to convert numpy numbers to Python native types, handling NaNs
    def to_native_numeric(val):
        if pd.isna(val):
            return None
        # Convert numpy numeric types to Python native types
        if isinstance(val, (int, float)):
            return val
        elif hasattr(val, 'item'): # For numpy scalars
            return val.item()
        return val # Return as is if not a known numeric type


    for i in tqdm(range(0, len(remaining_ids), 50), desc="videos.list"):
        batch = remaining_ids[i:i+50]

        try:
            response = api_call_with_retry(
                lambda b=batch: youtube.videos().list(
                    part="snippet,contentDetails,statistics",  # dropped 'status' — saves quota
                    id=",".join(b),
                    maxResults=50
                )
            )
        except Exception as e:
            print(f"  ✗ Batch {i//50} failed after retries: {e}. Progress saved.")
            save_checkpoint("video_details_rows", rows)
            raise

        for item in response.get("items", []):
            snippet = item.get("snippet", {})
            content = item.get("contentDetails", {})
            stats   = item.get("statistics", {})

            rows.append({
                "video_id":             item.get("id"),
                "actual_channel_id":    snippet.get("channelId"),
                "actual_channel_title": snippet.get("channelTitle"),
                "title":                snippet.get("title"),
                "actual_published_at":  snippet.get("publishedAt"),
                "duration_iso8601":     content.get("duration"),
                "duration_seconds":     parse_duration_to_seconds(content.get("duration")),
                "view_count":           to_native_numeric(pd.to_numeric(stats.get("viewCount"),   errors="coerce")),
                "like_count":           to_native_numeric(pd.to_numeric(stats.get("likeCount"),   errors="coerce")),
                "comment_count":        to_native_numeric(pd.to_numeric(stats.get("commentCount"), errors="coerce")),
                # privacy_status not returned (dropped 'status' part) — will be set to
                # 'public' downstream since only public videos appear in playlist results
                "privacy_status":       "public"
            })

        # ── Save after every batch ──
        save_checkpoint("video_details_rows", rows)
        time.sleep(0.05)  # 50 ms polite delay


    return pd.DataFrame(rows)

metadata_df = get_video_details(video_list_df["video_id"].tolist())

raw_df = video_list_df.merge(metadata_df, on="video_id", how="left")
raw_df = raw_df[raw_df["privacy_status"].fillna("public") == "public"].copy()

raw_df["actual_published_at"] = pd.to_datetime(
    raw_df["actual_published_at"], errors="coerce", utc=True
).dt.tz_localize(None)

start_dt = pd.Timestamp("2010-01-01")
end_dt   = pd.Timestamp("2011-12-31 23:59:59")

raw_df = raw_df[
    raw_df["actual_published_at"].notna() &
    (raw_df["actual_published_at"] >= start_dt) &
    (raw_df["actual_published_at"] <= end_dt)
].copy()

raw_df.to_csv(f"{OUTPUT_DIR}/youtube_video_level_raw.csv", index=False)

print(raw_df.shape)
raw_df.head()

  ↩  resuming from checkpoint → /content/drive/MyDrive/Classes/MIS 584/Project Colab Output/checkpoints/video_details_rows.json
  Total IDs: 1,270,267 | Already fetched: 1,270,302 | Remaining: 15


videos.list: 100%|██████████| 1/1 [00:24<00:00, 24.01s/it]

  ✓ checkpoint saved → /content/drive/MyDrive/Classes/MIS 584/Project Colab Output/checkpoints/video_details_rows.json


(82664, 14)


,channel_id,channel_title,video_id,playlist_published_at,actual_channel_id,actual_channel_title,title,actual_published_at,duration_iso8601,duration_seconds,view_count,like_count,comment_count,privacy_status
158,UC-8Q-hLdECwQmaWNwXitYDw,KatyPerryVEVO,LG1bLlGfirw,2011-12-28T08:00:00Z,UC-8Q-hLdECwQmaWNwXitYDw,KatyPerryVEVO,Katy Perry - The Making of “The One That Got A...,2011-12-28 08:00:00,PT5M30S,330.0,3.891596e+06,26118.0,1758.0,public
159,UC-8Q-hLdECwQmaWNwXitYDw,KatyPerryVEVO,Ahha3Cqe_fk,2011-11-11T18:37:24Z,UC-8Q-hLdECwQmaWNwXitYDw,KatyPerryVEVO,Katy Perry - The One That Got Away (Official M...,2011-11-11 18:37:24,PT4M50S,290.0,1.111077e+09,5160338.0,240865.0,public
160,UC-8Q-hLdECwQmaWNwXitYDw,KatyPerryVEVO,UhQsh6ciXuc,2011-11-04T13:00:00Z,UC-8Q-hLdECwQmaWNwXitYDw,KatyPerryVEVO,Katy Perry - The One That Got Away (Music Vide...,2011-11-04 13:00:00,PT36S,36.0,9.844218e+06,35166.0,7958.0,public
161,UC-8Q-hLdECwQmaWNwXitYDw,KatyPerryVEVO,ePQe5E69INg,2011-10-06T16:43:37Z,UC-8Q-hLdECwQmaWNwXitYDw,KatyPerryVEVO,Katy Perry - The One That Got Away,2011-10-06 16:43:37,PT3M49S,229.0,3.626739e+07,128698.0,6259.0,public
162,UC-8Q-hLdECwQmaWNwXitYDw,KatyPerryVEVO,CsYnQjJRf6o,2011-08-29T11:00:00Z,UC-8Q-hLdECwQmaWNwXitYDw,KatyPerryVEVO,Katy Perry - Making of “Last Friday Night (T.G...,2011-08-29 11:00:00,PT13M28S,808.0,1.017226e+07,69595.0,4782.0,public
